# История Казахстана QA-бот — Design C (Docling OCR + Unsloth/Colab + Gemini teacher)

Pipeline: **PDF (сканы) → Docling OCR → teacher LLM (Gemini) QA generation → embedding dedup → retrieval-grounding filter → Unsloth QLoRA SFT → eval (separate Gemini judge model) → push to HuggingFace Hub**.

Adjustments from the general Design C proposal, per project constraints:
- OCR: **Docling only** (no paid Mistral OCR API).
- Fine-tuning: **Unsloth on Colab's free T4 GPU**, adapter pushed to the HF Hub at the end.
- Teacher / judge LLM: **Gemini only** (free-tier API key from Google AI Studio). Three distinct models are used so no model grades its own output: `GEMINI_TEACHER_MODEL` generates the QA pairs, `GEMINI_JUDGE_MODEL` scores the fine-tuned model's answers, and the **student** being fine-tuned is a separate local model (`BASE_MODEL`, Qwen2.5-3B-Instruct).

**Before running:** Runtime → Change runtime type → T4 GPU. You'll need a Gemini API key and a HuggingFace **write** token, entered via Colab's Secrets panel (key icon in the left sidebar) as `GEMINI_API_KEY`, `HF_TOKEN`.


## 0. Setup

In [ ]:
# Core installs: OCR, fine-tuning stack, teacher-LLM SDK, embeddings, vector store
!pip install --upgrade --no-cache-dir pypdf docling
!pip install --upgrade --no-cache-dir unsloth unsloth_zoo trl peft accelerate bitsandbytes
!pip install --upgrade --no-cache-dir google-genai pydantic
!pip install --upgrade --no-cache-dir sentence-transformers lancedb
!pip install --upgrade --no-cache-dir wandb  # optional experiment tracking

# NOTE: pip may print a dependency-conflict warning here about `google-adk`
# (a package Colab's base image ships pre-installed) wanting an older
# opentelemetry-api/-sdk than wandb/google-genai need. It's safe to ignore —
# we never import google-adk in this notebook, and there is no single
# opentelemetry version that satisfies google-adk, wandb, and the otlp
# exporter simultaneously, so pinning one to please google-adk only breaks
# the others. The warning does not stop the install or affect anything below.


In [ ]:
import json
import re
import time
import getpass
from pathlib import Path

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pydantic import BaseModel, Field

print("Torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())


### Config

Everything project-specific lives in this one cell — change these, not the pipeline code.


In [ ]:
# --- Source document -------------------------------------------------------
PDF_PATH = "/content/history_kz_11.pdf"   # upload via the Colab file browser first
FIRST_PAGE = 3                             # 1-indexed, per the project brief
LAST_PAGE = 23

# --- Teacher / judge LLM (Gemini only) --------------------------------------
# Two distinct Gemini models so the judge never grades its own teacher's output.
GEMINI_TEACHER_MODEL = "gemini-3.1-flash-lite"  # Google's recommended pick for high-volume, low-reasoning tasks like this
GEMINI_JUDGE_MODEL = "gemini-3.5-flash"         # stronger reasoning for scoring accuracy/faithfulness

# --- Base model for fine-tuning (student) -----------------------------------
BASE_MODEL = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 2048

# --- Dataset targets ---------------------------------------------------------
QA_PER_CHUNK = 4          # one per question type, per chunk
DEDUP_SIM_THRESHOLD = 0.92     # drop near-duplicate questions above this cosine sim
GROUNDING_SIM_THRESHOLD = 0.55  # drop QA pairs whose answer doesn't match retrieved chunk

# --- HuggingFace Hub target --------------------------------------------------
HF_USERNAME = "zhadyrazhan"           # <-- set this
HF_REPO_NAME = "kz-history-qwen2.5-3b-lora"
HF_REPO_ID = f"{HF_USERNAME}/{HF_REPO_NAME}"

# --- Experiment tracking ------------------------------------------------------
USE_WANDB = False   # flip to True if you've set up a WANDB_API_KEY secret

SEED = 42


In [ ]:
# API keys / tokens: prefer Colab Secrets (key icon in the sidebar), fall back to a prompt
try:
    from google.colab import userdata
    def get_secret(name):
        try:
            return userdata.get(name)
        except Exception:
            return None
except ImportError:
    def get_secret(name):
        return None

GEMINI_API_KEY = (get_secret("GEMINI_API_KEY") or getpass.getpass("GEMINI_API_KEY: ")).strip()
HF_TOKEN = (get_secret("HF_TOKEN") or getpass.getpass("HF_TOKEN (write access): ")).strip()

for _name, _value in [("GEMINI_API_KEY", GEMINI_API_KEY), ("HF_TOKEN", HF_TOKEN)]:
    if not _value.isascii():
        raise ValueError(
            f"{_name} contains a non-ASCII character — it got corrupted somewhere in copy/paste "
            "(smart quotes, invisible unicode, etc). Re-copy it fresh from its source (don't retype) "
            "and, if it's stored in Colab Secrets, delete and re-add that secret before rerunning."
        )

from huggingface_hub import login as hf_login
hf_login(token=HF_TOKEN)
print("HF login OK")


## 1. OCR extraction (Docling)

We first slice out only pages 3–23 with `pypdf` (Docling doesn't need to touch the rest of the book), then run Docling's OCR + layout model on that slice and export clean Markdown.


In [ ]:
from pypdf import PdfReader, PdfWriter

reader = PdfReader(PDF_PATH)
writer = PdfWriter()
for page_num in range(FIRST_PAGE - 1, LAST_PAGE):   # pypdf pages are 0-indexed
    writer.add_page(reader.pages[page_num])

SLICED_PDF_PATH = "/content/history_kz_11_slice.pdf"
with open(SLICED_PDF_PATH, "wb") as f:
    writer.write(f)

print(f"Sliced pages {FIRST_PAGE}-{LAST_PAGE} -> {SLICED_PDF_PATH}")


In [ ]:
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import EasyOcrOptions, PdfPipelineOptions
from docling.document_converter import DocumentConverter, PdfFormatOption

# Docling's default OCR language list is en/fr/de/es — it silently drops
# Cyrillic-only letters (keeping just the ones that look Latin, e.g. О, Х, Т, В),
# which is why the first OCR pass came out mostly blank. Force Russian OCR,
# keeping English too since the textbook mixes in some English glossary terms.
ocr_options = EasyOcrOptions(lang=["ru", "en"])
pipeline_options = PdfPipelineOptions(do_ocr=True, ocr_options=ocr_options)

converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)
    }
)
result = converter.convert(SLICED_PDF_PATH)
full_markdown = result.document.export_to_markdown()

print(f"Extracted {len(full_markdown)} characters of Markdown\n")
print("--- OCR output fragment (first 800 chars) ---")
print(full_markdown[:800])


In [ ]:
# Required deliverable: history_text.txt
with open("history_text.txt", "w", encoding="utf-8") as f:
    f.write(full_markdown)

print("Saved history_text.txt")


## 2. Chunking

Split the Markdown into section-sized chunks so the teacher LLM sees focused context per call (better grounding than one giant prompt), and so we have retrievable units for the grounding-filter step later.


In [ ]:
def chunk_markdown(markdown_text: str, target_chars: int = 1200) -> list[dict]:
    """Split on Markdown headings first, then hard-wrap any oversized section."""
    sections = re.split(r"\n(?=#{1,3} )", markdown_text)
    chunks = []
    for section in sections:
        section = section.strip()
        if not section:
            continue
        if len(section) <= target_chars:
            chunks.append(section)
        else:
            for i in range(0, len(section), target_chars):
                chunks.append(section[i : i + target_chars])
    return [{"chunk_id": i, "text": c} for i, c in enumerate(chunks)]

chunks = chunk_markdown(full_markdown)
print(f"Built {len(chunks)} chunks")
print("\n--- sample chunk ---\n")
print(chunks[0]["text"][:500])


## 3. QA generation (teacher LLM, structured output)

We ask the teacher for a strict JSON batch (validated against a Pydantic schema) per chunk, covering the brief's four question types: "Кто такой...?", "Что написал...?", "В каком веке...?", "Какое значение имеет...?". Structured output means no manual `json.loads`-and-hope-it-parses — invalid responses are rejected and retried.


In [ ]:
class QAPair(BaseModel):
    question: str = Field(description="Вопрос на русском языке")
    answer: str = Field(description="Точный, фактологичный ответ на основе текста")
    question_type: str = Field(description="Один из: кто_такой, что_написал, век, значение")

class QAPairsBatch(BaseModel):
    pairs: list[QAPair]

QA_PROMPT_TEMPLATE = """Ты — эксперт по истории Казахстана, готовящий вопросы для 11 класса. На основе ТОЛЬКО следующего фрагмента учебника составь {n} пар вопрос-ответ.

Используй разные типы вопросов: "Кто такой...?", "Что написал/создал...?", "В каком веке...?", "Какое значение имеет...?".

Требования:
- Ответ должен быть проверяемым фактом ИЗ ЭТОГО ФРАГМЕНТА, не придумывай ничего.
- Если во фрагменте нет достаточно фактов для {n} разных вопросов — верни меньше пар.
- Ответы на русском языке, 1-3 предложения.

Фрагмент учебника:
{chunk}
"""


In [ ]:
# --- Gemini teacher backend --------------------------------------------------
class TeacherQuotaExhausted(Exception):
    """Raised when the Gemini API reports the request quota is used up —
    further retries would only burn through an already-exhausted daily limit."""

def _generate_teacher(prompt: str) -> QAPairsBatch:
    from google import genai
    from google.genai import types

    client = genai.Client(api_key=GEMINI_API_KEY)
    response = client.models.generate_content(
        model=GEMINI_TEACHER_MODEL,
        contents=prompt,
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=QAPairsBatch,
        ),
    )
    return QAPairsBatch.model_validate_json(response.text)

def generate_qa_for_chunk(chunk_text: str, n: int = QA_PER_CHUNK, retries: int = 2) -> list[dict]:
    prompt = QA_PROMPT_TEMPLATE.format(n=n, chunk=chunk_text)

    for attempt in range(retries + 1):
        try:
            batch = _generate_teacher(prompt)
            return [p.model_dump() for p in batch.pairs]
        except Exception as e:
            if "RESOURCE_EXHAUSTED" in str(e):
                raise TeacherQuotaExhausted(str(e)) from e
            if attempt == retries:
                print(f"  [warn] chunk failed after {retries + 1} attempts: {e}")
                return []
            time.sleep(2 * (attempt + 1))


In [ ]:
raw_qa_pairs = []
for i, chunk in enumerate(chunks):
    try:
        pairs = generate_qa_for_chunk(chunk["text"])
    except TeacherQuotaExhausted as e:
        print(f"[stop] Gemini quota exhausted after {i}/{len(chunks)} chunks: {e}\n"
              "Wait for the daily quota to reset (or switch GEMINI_TEACHER_MODEL / upgrade "
              "your plan), then rerun this cell — already-generated pairs are kept below.")
        break
    for p in pairs:
        p["chunk_id"] = chunk["chunk_id"]
    raw_qa_pairs.extend(pairs)
    time.sleep(0.5)  # gentle on free-tier rate limits

print(f"Generated {len(raw_qa_pairs)} raw QA pairs from {len(chunks)} chunks")
print("\n--- sample generated pairs ---")
for p in raw_qa_pairs[:5]:
    print(f"Q: {p['question']}\nA: {p['answer']}\n")


## 4. Embedding-based deduplication

The brief's suggested "generate templates and replicate" approach risks near-duplicate questions. We embed every question with a multilingual model (BGE-M3 handles Russian well) and greedily drop anything too similar to a pair we've already kept.


In [ ]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("BAAI/bge-m3")

questions = [p["question"] for p in raw_qa_pairs]
q_embeddings = embedder.encode(questions, normalize_embeddings=True, show_progress_bar=True)


In [ ]:
def dedup_by_embedding(pairs: list[dict], embeddings: np.ndarray, threshold: float) -> list[dict]:
    kept_pairs, kept_embeddings = [], []
    for pair, emb in zip(pairs, embeddings):
        if kept_embeddings:
            sims = np.dot(np.array(kept_embeddings), emb)
            if sims.max() >= threshold:
                continue  # too similar to something we already kept
        kept_pairs.append(pair)
        kept_embeddings.append(emb)
    return kept_pairs

deduped_pairs = dedup_by_embedding(raw_qa_pairs, q_embeddings, DEDUP_SIM_THRESHOLD)
print(f"Deduped: {len(raw_qa_pairs)} -> {len(deduped_pairs)} pairs "
      f"({len(raw_qa_pairs) - len(deduped_pairs)} near-duplicates dropped)")


## 5. Retrieval-grounding filter (LanceDB)

Teacher LLMs still hallucinate occasionally, even when told to stick to the given text. We index all chunks in LanceDB and, for each surviving QA pair, check that its answer is actually close (in embedding space) to *some* real chunk — not just the one it claims to come from. Pairs that don't ground anywhere get dropped before they can poison the SFT set.


In [ ]:
import lancedb

chunk_texts = [c["text"] for c in chunks]
chunk_embeddings = embedder.encode(chunk_texts, normalize_embeddings=True, show_progress_bar=True)

db = lancedb.connect("/content/lancedb")
table = db.create_table(
    "textbook_chunks",
    data=[
        {"chunk_id": c["chunk_id"], "text": c["text"], "vector": emb.tolist()}
        for c, emb in zip(chunks, chunk_embeddings)
    ],
    mode="overwrite",
)
print(f"Indexed {table.count_rows()} chunks in LanceDB")


In [ ]:
answer_embeddings = embedder.encode(
    [p["answer"] for p in deduped_pairs], normalize_embeddings=True, show_progress_bar=True
)

grounded_pairs = []
for pair, emb in zip(deduped_pairs, answer_embeddings):
    hit = table.search(emb.tolist()).limit(1).to_list()[0]
    score = 1 - hit["_distance"] / 2  # cosine distance -> similarity, roughly
    if score >= GROUNDING_SIM_THRESHOLD:
        pair["grounding_score"] = round(float(score), 3)
        pair["grounded_chunk_id"] = hit["chunk_id"]
        grounded_pairs.append(pair)

print(f"Grounding filter: {len(deduped_pairs)} -> {len(grounded_pairs)} pairs kept "
      f"({len(deduped_pairs) - len(grounded_pairs)} likely-hallucinated pairs dropped)")


## 6. Save the SFT dataset

Required deliverable: `history_sft_dataset.json`. Format matches the project brief's instruction/output schema.


In [ ]:
sft_dataset = [
    {"instruction": p["question"], "output": p["answer"], "question_type": p["question_type"]}
    for p in grounded_pairs
]

with open("history_sft_dataset.json", "w", encoding="utf-8") as f:
    json.dump(sft_dataset, f, ensure_ascii=False, indent=2)

print(f"Saved history_sft_dataset.json with {len(sft_dataset)} QA pairs")
if len(sft_dataset) < 100:
    print("[warn] under the 100-pair minimum from the grading rubric — "
          "consider lowering GROUNDING_SIM_THRESHOLD or adding more source pages")
elif len(sft_dataset) < 200:
    print("[note] under the 200-pair 'мало данных?' suggestion — fine, but more helps quality")


## 7. Load base model + LoRA adapters (Unsloth)


In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    random_state=SEED,
)

print("Model + LoRA adapters loaded")


## 8. Baseline test (before fine-tuning)

Required deliverable: show the model's answers *before* training, for comparison.


In [ ]:
PROMPT_TEMPLATE = "### Instruction:\n{instruction}\n\n### Response:\n"

TEST_QUESTIONS = [
    "Кто такой Махмуд Кашгари?",
    "Какую книгу написал Юсуф Баласагуни?",
    "Что такое Кутадгу билиг?",
    "Кто такой Ходжа Ахмед Яссауи?",
    "В каком веке жил Юсуф Баласагуни?",
]

def ask(question: str, max_new_tokens: int = 200) -> str:
    prompt = PROMPT_TEMPLATE.format(instruction=question)
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=max_new_tokens)
    input_len = inputs.input_ids.shape[1]
    return tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)


In [ ]:
FastLanguageModel.for_inference(model)

print("Результат ДО fine-tuning:\n" + "=" * 50)
baseline_answers = {}
for q in TEST_QUESTIONS:
    a = ask(q)
    baseline_answers[q] = a
    print(f"Q: {q}\nA: {a}\n")


## 9. Prepare the training dataset

In [ ]:
from datasets import Dataset

with open("history_sft_dataset.json", "r", encoding="utf-8") as f:
    sft_data = json.load(f)

formatted_texts = [
    PROMPT_TEMPLATE.format(instruction=item["instruction"]) + item["output"] + tokenizer.eos_token
    for item in sft_data
]

train_dataset = Dataset.from_dict({"text": formatted_texts})
print(train_dataset)
print("\n--- sample formatted example ---\n")
print(formatted_texts[0])


## 10. Train (Unsloth + TRL SFTTrainer)

`num_train_epochs` scales with dataset size rather than a fixed `max_steps`, since our dataset (post-dedup/grounding) will vary in size run to run.


In [ ]:
if USE_WANDB:
    import wandb
    wandb.login()
    report_to = "wandb"
else:
    report_to = "none"


In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

FastLanguageModel.for_training(model)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,       # effective batch size = 8
        warmup_steps=10,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        output_dir="outputs",
        report_to=report_to,
        seed=SEED,
    ),
)

print("Запуск обучения LoRA (следите за снижением loss)...")
trainer_stats = trainer.train()
print("Обучение завершено!")


## 11. Loss curve

Required deliverable: a visible plot of training loss decreasing over steps.


In [ ]:
log_history = trainer.state.log_history
steps = [entry["step"] for entry in log_history if "loss" in entry]
losses = [entry["loss"] for entry in log_history if "loss" in entry]

plt.figure(figsize=(8, 4))
plt.plot(steps, losses, marker="o")
plt.xlabel("Training step")
plt.ylabel("Loss")
plt.title("SFT training loss")
plt.grid(True, alpha=0.3)
plt.savefig("loss_curve.png", dpi=120, bbox_inches="tight")
plt.show()


## 12. Test after fine-tuning (5+ examples)

Required deliverable: at least 5 example answers from the fine-tuned model.


In [ ]:
FastLanguageModel.for_inference(model)

print("Результат ПОСЛЕ fine-tuning:\n" + "=" * 50)
finetuned_answers = {}
for q in TEST_QUESTIONS:
    a = ask(q)
    finetuned_answers[q] = a
    print(f"Q: {q}\nA: {a}\n")


## 13. LLM-as-judge evaluation

Score each before/after answer pair on accuracy and faithfulness with the same teacher LLM, using a typed rubric instead of eyeballing the outputs.


In [ ]:
class JudgeScore(BaseModel):
    accuracy: int = Field(ge=1, le=5, description="Фактическая точность ответа (1-5)")
    faithfulness: int = Field(ge=1, le=5, description="Соответствие учебнику, без выдумок (1-5)")
    format_ok: bool = Field(description="Ответ полный и на русском языке")
    comment: str = Field(description="Краткое обоснование оценки")

JUDGE_PROMPT = """Ты — строгий преподаватель истории Казахстана, оценивающий ответ ученика.

Вопрос: {question}
Эталонный фрагмент учебника (может быть неполным): {context}
Ответ модели: {answer}

Оцени точность (accuracy) и соответствие источнику (faithfulness) от 1 до 5.
"""

def judge_answer(question: str, answer: str, context: str = "") -> JudgeScore:
    # Uses GEMINI_JUDGE_MODEL, a different model from GEMINI_TEACHER_MODEL that
    # generated the QA pairs, so the judge isn't grading its own outputs.
    from google import genai
    from google.genai import types

    prompt = JUDGE_PROMPT.format(question=question, context=context[:600], answer=answer)
    client = genai.Client(api_key=GEMINI_API_KEY)
    response = client.models.generate_content(
        model=GEMINI_JUDGE_MODEL,
        contents=prompt,
        config=types.GenerateContentConfig(
            response_mime_type="application/json", response_schema=JudgeScore
        ),
    )
    return JudgeScore.model_validate_json(response.text)


In [ ]:
judge_rows = []
for q in TEST_QUESTIONS:
    score = judge_answer(q, finetuned_answers[q])
    judge_rows.append({
        "question": q,
        "answer": finetuned_answers[q],
        "accuracy": score.accuracy,
        "faithfulness": score.faithfulness,
        "format_ok": score.format_ok,
        "comment": score.comment,
    })

judge_df = pd.DataFrame(judge_rows)
display(judge_df)
print(f"\nMean accuracy: {judge_df['accuracy'].mean():.2f} / 5")
print(f"Mean faithfulness: {judge_df['faithfulness'].mean():.2f} / 5")


## 14. Save adapter + push to HuggingFace Hub


In [ ]:
LOCAL_ADAPTER_DIR = "kz_history_lora"
model.save_pretrained(LOCAL_ADAPTER_DIR)
tokenizer.save_pretrained(LOCAL_ADAPTER_DIR)
print(f"LoRA adapter saved locally to ./{LOCAL_ADAPTER_DIR}")


In [ ]:
model.push_to_hub(HF_REPO_ID)
tokenizer.push_to_hub(HF_REPO_ID)
print(f"\nAdapter published: https://huggingface.co/{HF_REPO_ID}")


## Optional: merge + GGUF export for local serving (Ollama)

Not required for grading — useful if you want to demo the bot outside Colab, without a live GPU session, via `ollama run kz-history`. Uncomment to run (adds a few minutes).

```python
# model.save_pretrained_gguf(
#     "kz_history_gguf",
#     tokenizer,
#     quantization_method="q4_k_m",
# )
# # Then locally: `ollama create kz-history -f Modelfile` pointing at the .gguf file,
# # per Ollama's docs for importing a GGUF model.
```


## Deliverables checklist

- [x] `history_finetuning.ipynb` — this notebook, run top-to-bottom with outputs saved
- [x] `history_text.txt` — saved in section 1
- [x] `history_sft_dataset.json` — saved in section 6
- [x] OCR output fragment visible — section 1
- [x] Sample generated QA pairs visible — section 3
- [x] Loss curve visible — section 11
- [x] 5+ example model answers visible — section 12
- [x] LoRA adapter pushed to HuggingFace Hub — section 14
